In [1]:
import pandas as pd

In [2]:
# Leer el archivo copiado
df = pd.read_parquet("sp500_history_copy.parquet")

In [3]:
#Explora las columnas disponibles y sus tipos.
df.columns.tolist()
df.dtypes

date                datetime64[ns]
symbol                         str
assetid                      int64
security_name                  str
sector                         str
industry                       str
subsector                      str
in_sp500                     int32
open                       float32
high                       float32
low                        float32
close                      float32
volume                     float32
unadjusted_close           float32
dtype: object

In [18]:
#Muestra una vista rapida (head) y un resumen de memoria.
df.head()


,date,symbol,assetid,security_name,sector,industry,subsector,in_sp500,open,high,low,close,volume,unadjusted_close
3803,2015-01-02,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1,37.621143,37.739910,36.881145,37.054726,1675607.25,40.560001
3804,2015-01-05,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1,36.835464,36.963367,36.269047,36.360405,2235430.25,39.799999
3805,2015-01-06,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1,36.369541,36.561394,35.647816,35.793987,2281755.75,39.180000
3806,2015-01-07,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1,36.104603,36.369541,35.894482,36.269047,3677474.50,39.700001
3807,2015-01-08,A,131684,Agilent Technologies Inc Common,Health Care,Life Sciences Tools & Services,Life Sciences Tools & Services,1,36.762383,37.438427,36.707565,37.356205,2316541.00,40.889999


In [5]:
#Validaciones iniciales

# Rangos de fechas
print("Rango de fechas:")   
print(f"Fecha mínima: {df['date'].min()}, Fecha máxima: {df['date'].max()}")

Rango de fechas:
Fecha mínima: 1990-01-02 00:00:00, Fecha máxima: 2026-01-30 00:00:00


In [9]:
# Duplicados exactos por fecha y ticker (misma fecha + mismo símbolo)
counts = df.groupby(['date', 'symbol']).size()
dup_exact = counts[counts > 1]

if dup_exact.empty:
    print("No hay duplicados exactos (fecha, ticker).")
else:
    print(f"Hay {len(dup_exact)} duplicados exactos (fecha, ticker).")
    print(dup_exact.head(20))

# Múltiples símbolos para la misma empresa en la misma fecha
# Usamos 'security_name' como proxy de empresa
company_col = 'security_name'

sym_nunique = df.groupby(['date', company_col])['symbol'].nunique()
multisym = sym_nunique[sym_nunique > 1]

if multisym.empty:
    print("No hay empresas con múltiples símbolos en la misma fecha.")
else:
    print(f"Hay {len(multisym)} combinaciones (fecha, empresa) con múltiples símbolos.")
    symbols_per = (df.groupby(['date', company_col])['symbol']
                     .apply(lambda x: ', '.join(sorted(set(x)))))
    detalle = symbols_per[multisym.index].reset_index(name='symbols')
    detalle['n_symbols'] = multisym.values
    print(detalle.head(20).to_string(index=False))


No hay duplicados exactos (fecha, ticker).
No hay empresas con múltiples símbolos en la misma fecha.


In [ ]:
# Resolver duplicados por date + assetid:
# conservar la fila con mayor volume.
entity_keys = ['date', 'assetid']

dup_mask = df.duplicated(subset=entity_keys, keep=False)
rows_in_dup_groups = int(dup_mask.sum())
groups_with_dups = int(df.loc[dup_mask].groupby(entity_keys).ngroups)

print(f"Filas en grupos duplicados (date, assetid): {rows_in_dup_groups}")
print(f"Grupos duplicados (date, assetid): {groups_with_dups}")

if rows_in_dup_groups > 0:
    # Orden estable para desempates: mayor volumen, luego symbol asc
    df = (
        df.sort_values(['date', 'assetid', 'volume', 'symbol'], ascending=[True, True, False, True], na_position='last')
          .drop_duplicates(subset=entity_keys, keep='first')
          .reset_index(drop=True)
    )

remaining_dups = int(df.duplicated(subset=entity_keys).sum())
print(f"Duplicados restantes (date, assetid): {remaining_dups}")

# Validacion adicional: mismo date + symbol repetido
dup_symbol = int(df.duplicated(subset=['date', 'symbol']).sum())
print(f"Duplicados exactos restantes (date, symbol): {dup_symbol}")


Filas en grupos duplicados (date, assetid): 0
Grupos duplicados (date, assetid): 0
Duplicados restantes (date, assetid): 0
Duplicados exactos restantes (date, symbol): 0


In [13]:
# Filtrar para que dato de inicio sea 1 de enero de 2015
df = df[df['date'] >= '2015-01-01']
print(f"Rango de fechas después del filtro: {df['date'].min()} a {df['date'].max()}")

Rango de fechas después del filtro: 2015-01-02 00:00:00 a 2026-01-30 00:00:00


In [14]:
df.to_pickle("sp500_history_filtered.pkl")